# Uji Model Embedding untuk RAG Chatbot Regulasi

Tujuan (masukan Kak Wiwi): membandingkan beberapa model multilingual dari sisi
1. waktu loading
2. tambahan memori (RAM)
3. kecepatan embedding (per query dan per batch dokumen)
4. kualitas pencarian (Recall@k dan MRR)
5. jumlah dimensi vektor (menentukan ukuran kolom `vector(n)` di Supabase pgvector)

**Cara pakai:** jalankan sel dari atas ke bawah. Di bagian 3, data contoh dengan potongan pasal/ayat asli dari Excel dan 20-30 pertanyaan uji.

## 1. Instalasi

In [1]:
!pip install -q sentence-transformers psutil pandas openpyxl

## 2. Import dan daftar model kandidat

Model E5 wajib memakai prefix `query: ` dan `passage: `. Model MiniLM tidak memakai prefix.

In [2]:
import gc, os, time
import numpy as np
import pandas as pd
import psutil
from sentence_transformers import SentenceTransformer

proc = psutil.Process(os.getpid())

KANDIDAT = {
    'intfloat/multilingual-e5-small': {'query_prefix': 'query: ', 'doc_prefix': 'passage: '},
    'intfloat/multilingual-e5-base': {'query_prefix': 'query: ', 'doc_prefix': 'passage: '},
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2': {'query_prefix': '', 'doc_prefix': ''},
}

## 3. Data uji

- `corpus`: diambil dari `edu_knowledge_base.xlsx` (kategori + pasal + teks pasal).
- `queries`: dari `pertanyaan_uji.xlsx` (30 pertanyaan bahasa sehari-hari beserta id pasal yang seharusnya muncul).

In [5]:
from google.colab import files
files.upload()

Saving edu_knowledge_base.xlsx to edu_knowledge_base.xlsx


{'edu_knowledge_base.xlsx': b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00b\xee\x9dh^\x01\x00\x00\x90\x04\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0

In [6]:
# Upload 2 file ke Colab: edu_knowledge_base.xlsx dan pertanyaan_uji.xlsx
df = pd.read_excel('edu_knowledge_base.xlsx')
df = df.dropna(subset=['teks_pasal']).reset_index(drop=True)

# id dibuat urut P01, P02, ... sesuai urutan baris di Excel
corpus = []
for i, r in df.iterrows():
    teks = f"{r['kategori']}. {r['pasal']}. {r['teks_pasal']}"
    corpus.append({'id': f'P{i+1:02d}', 'teks': teks})

dfq = pd.read_excel('pertanyaan_uji.xlsx', sheet_name='pertanyaan_uji')
queries = [{'q': r['pertanyaan'], 'relevan': [x.strip() for x in str(r['id_relevan']).split(',')]}
           for _, r in dfq.iterrows()]

print(len(corpus), 'dokumen,', len(queries), 'pertanyaan')

9 dokumen, 30 pertanyaan


## 4. Fungsi evaluasi

In [7]:
def evaluasi(nama, cfg, corpus, queries, ks=(1, 3, 5)):
    gc.collect()
    ram0 = proc.memory_info().rss / 1e6

    t0 = time.time()
    model = SentenceTransformer(nama)
    t_load = time.time() - t0
    ram_model = proc.memory_info().rss / 1e6 - ram0

    ids = [c['id'] for c in corpus]
    docs = [cfg['doc_prefix'] + c['teks'] for c in corpus]

    t0 = time.time()
    D = model.encode(docs, normalize_embeddings=True, batch_size=16)
    t_corpus = time.time() - t0

    # warm-up supaya latency query tidak ikut menghitung inisialisasi
    model.encode([cfg['query_prefix'] + 'warm up'], normalize_embeddings=True)

    latensi, hits, rr = [], {k: 0 for k in ks}, 0.0
    for item in queries:
        t0 = time.time()
        qv = model.encode([cfg['query_prefix'] + item['q']], normalize_embeddings=True)[0]
        latensi.append(time.time() - t0)

        skor = D @ qv
        urut = [ids[i] for i in np.argsort(-skor)]
        for k in ks:
            if set(urut[:k]) & set(item['relevan']):
                hits[k] += 1
        for pos, d in enumerate(urut, 1):
            if d in item['relevan']:
                rr += 1 / pos
                break

    n = len(queries)
    hasil = {
        'model': nama.split('/')[-1],
        'dimensi': int(D.shape[1]),
        'loading (detik)': round(t_load, 2),
        'tambahan RAM (MB)': round(ram_model),
        'encode korpus (detik)': round(t_corpus, 2),
        'rata-rata 1 query (ms)': round(np.mean(latensi) * 1000, 1),
        'MRR': round(rr / n, 3),
    }
    for k in ks:
        hasil[f'Recall@{k}'] = round(hits[k] / n, 3)

    del model
    gc.collect()
    return hasil

## 5. Jalankan perbandingan

Catatan: pengukuran RAM di notebook adalah perkiraan kasar karena model dimuat berurutan di proses yang sama. Untuk angka yang lebih bersih, restart runtime lalu jalankan satu model per sesi.

In [8]:
semua = []
for nama, cfg in KANDIDAT.items():
    print('Menguji', nama, '...')
    semua.append(evaluasi(nama, cfg, corpus, queries))

tabel = pd.DataFrame(semua)
tabel

Menguji intfloat/multilingual-e5-small ...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (33 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,model,dimensi,loading (detik),tambahan RAM (MB),encode korpus (detik),rata-rata 1 query (ms),MRR,Recall@1,Recall@3,Recall@5
0,multilingual-e5-small,384,16.03,417,1.36,58.3,0.855,0.800,0.933,0.967
1,multilingual-e5-base,768,37.30,115,3.00,110.4,0.882,0.800,0.933,1.000
2,paraphrase-multilingual-MiniLM-L12-v2,384,11.24,15,0.85,32.7,0.776,0.633,0.900,0.967


## 6. Simpan hasil dan cara membaca

- **Recall@k**: persentase pertanyaan yang jawaban benarnya masuk k hasil teratas. Makin tinggi makin bagus.
- **MRR**: rata-rata 1/peringkat jawaban benar. Makin dekat ke 1 makin bagus.
- **Loading dan RAM**: menentukan waktu start server dan spesifikasi hosting FastAPI.
- **Dimensi**: ukuran kolom `vector(n)` di Supabase pgvector (pilih model dulu, baru buat tabel).

Pilih model dengan kualitas cukup (Recall@3 tinggi) tanpa membebani waktu start dan memori.

In [9]:
tabel.to_csv('hasil_uji_embedding.csv', index=False)
print('Tersimpan: hasil_uji_embedding.csv')

Tersimpan: hasil_uji_embedding.csv


In [10]:
# Tampilkan pertanyaan yang SALAH di urutan 1 untuk satu model
def cek_gagal(nama, cfg, corpus, queries, top=3):
    model = SentenceTransformer(nama)
    ids = [c['id'] for c in corpus]
    label = {c['id']: c['teks'].split('. ')[1] if '. ' in c['teks'] else c['id'] for c in corpus}  # nama pasal
    D = model.encode([cfg['doc_prefix'] + c['teks'] for c in corpus], normalize_embeddings=True)

    baris = []
    for item in queries:
        qv = model.encode([cfg['query_prefix'] + item['q']], normalize_embeddings=True)[0]
        skor = D @ qv
        urut = list(np.argsort(-skor))
        teratas = [ids[i] for i in urut[:top]]
        if teratas[0] not in item['relevan']:
            baris.append({
                'pertanyaan': item['q'],
                'seharusnya': ', '.join(f"{r} ({label[r]})" for r in item['relevan']),
                'malah muncul #1': f"{teratas[0]} ({label[teratas[0]]})",
                'top-3 ada yang benar?': 'ya' if set(teratas) & set(item['relevan']) else 'TIDAK',
            })
    hasil = pd.DataFrame(baris)
    print(f"{nama}: {len(hasil)} dari {len(queries)} pertanyaan salah di urutan 1")
    return hasil

pd.set_option('display.max_colwidth', None)
gagal_small = cek_gagal('intfloat/multilingual-e5-small', KANDIDAT['intfloat/multilingual-e5-small'], corpus, queries)
gagal_small

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

intfloat/multilingual-e5-small: 6 dari 30 pertanyaan salah di urutan 1


,pertanyaan,seharusnya,malah muncul #1,top-3 ada yang benar?
0,"Penagih menghubungi keluarga dan kontak darurat saya, boleh?",P07 (Pasal 62 ayat (1) dan (2)),P04 (Pasal 4 ayat (5)),TIDAK
1,"Penagih menekan saya dengan kata-kata kasar, bagaimana aturannya?",P07 (Pasal 62 ayat (1) dan (2)),P04 (Pasal 4 ayat (5)),ya
2,"Pinjol memanfaatkan saya yang sedang butuh uang mendesak, boleh?",P09 (Pasal 44 ayat (3) dan (4)),P01 (Pasal 3 ayat (2)),ya
3,Apakah pinjol wajib transparan soal informasi produknya?,P01 (Pasal 3 ayat (2)),P06 (Pasal 19 ayat (1) & Pasal 22 ayat (1) dan (2)),ya
4,Ke mana saya bisa mengadu kalau ada masalah dengan pinjol?,"P01 (Pasal 3 ayat (2)), P05 (Pasal 8 ayat (2))",P04 (Pasal 4 ayat (5)),TIDAK
5,Apa aturan soal pembuatan perjanjian pinjaman?,"P05 (Pasal 8 ayat (2)), P09 (Pasal 44 ayat (3) dan (4))",P06 (Pasal 19 ayat (1) & Pasal 22 ayat (1) dan (2)),ya


In [11]:
# Bandingkan: teks hukum saja vs penafsiran + teks hukum (e5-small)
dfp = pd.read_excel('edu_knowledge_base_penafsiran.xlsx').dropna(subset=['teks_pasal']).reset_index(drop=True)

corpus_v2 = []
for i, r in dfp.iterrows():
    teks = f"{r['kategori']}. {r['pasal']}. {r['penafsiran (pasal dengan bahasa yang lebih mudah dipahami)']} {r['teks_pasal']}"
    corpus_v2.append({'id': f'P{i+1:02d}', 'teks': teks})

nama = 'intfloat/multilingual-e5-small'
cfg = KANDIDAT[nama]
v1 = evaluasi(nama, cfg, corpus, queries);    v1['versi'] = 'teks hukum saja'
v2 = evaluasi(nama, cfg, corpus_v2, queries); v2['versi'] = 'penafsiran + teks hukum'
pd.DataFrame([v1, v2])[['versi', 'MRR', 'Recall@1', 'Recall@3', 'Recall@5']]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,versi,MRR,Recall@1,Recall@3,Recall@5
0,teks hukum saja,0.855,0.8,0.933,0.967
1,penafsiran + teks hukum,0.940,0.9,0.967,1.000
